In [7]:
import sys

print(sys.executable)
print(sys.version)

c:\Users\ronal\source\repos\ARTEFACT-DesafioTecnico\.venv\Scripts\python.exe
3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]


In [8]:
pip install rank-bm25

Note: you may need to restart the kernel to use updated packages.


In [9]:
from pathlib import Path
import json
import re
import unicodedata

from rank_bm25 import BM25Okapi

In [10]:
def normalize_text(text: str) -> str:
    """
    Normaliza o texto para busca BM25.

    - converte para minúsculas
    - remove acentos
    - mantém apenas palavras/números
    """
    text = text.lower()

    # Remove acentos
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        char
        for char in text
        if not unicodedata.combining(char)
    )

    return text


def tokenize(text: str) -> list[str]:
    """
    Converte um texto em tokens para o BM25.
    """
    text = normalize_text(text)

    return re.findall(r"\b\w+\b", text)


def load_chunks(chunks_dir: Path) -> list[dict]:
    """
    Carrega todos os chunks dos arquivos JSONL.
    """

    chunks = []

    for jsonl_path in chunks_dir.glob("*.jsonl"):
        print(f"Carregando: {jsonl_path.name}")

        with jsonl_path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()

                if not line:
                    continue

                chunk = json.loads(line)
                chunks.append(chunk)

    return chunks


def build_bm25(chunks: list[dict]):
    """
    Cria o índice BM25 a partir dos chunks.
    """

    corpus = [
        tokenize(chunk["text"])
        for chunk in chunks
    ]

    return BM25Okapi(corpus)


def search_bm25(
    bm25,
    chunks: list[dict],
    query: str,
    top_k: int = 5,
):
    """
    Executa uma busca BM25 e retorna os chunks mais relevantes.
    """

    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)

    # Índices ordenados pela maior pontuação
    ranked_indexes = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True,
    )

    results = []

    for index in ranked_indexes[:top_k]:
        chunk = chunks[index].copy()
        chunk["score"] = float(scores[index])

        results.append(chunk)

    return results

In [14]:
if __name__ == "__main__":

    chunks_dir = Path("data/chunks")

    # 1. Carrega todos os chunks
    chunks = load_chunks(chunks_dir)

    print()
    print(f"Total de chunks: {len(chunks)}")

    # 2. Cria índice BM25
    bm25 = build_bm25(chunks)

    print("Índice BM25 criado.")

    # 3. Teste
    query = "quais são as formas de pagamento?"

    results = search_bm25(
        bm25,
        chunks,
        query,
        top_k=5,
    )

    print()
    print(f"Consulta: {query}")
    print("=" * 80)

    for result in results:
        print(
            f"\nScore: {result['score']:.4f}"
        )
        print(
            f"Documento: {result['source']}"
        )
        print(
            f"Página: {result['page']}"
        )
        print(
            f"Chunk: {result['chunk_id']}"
        )
        #print(
        #    f"Texto:\n{result['text'][:500]}"
        #)

Carregando: politicas_da_loja.jsonl

Total de chunks: 8
Índice BM25 criado.

Consulta: quais são as formas de pagamento?

Score: 6.1985
Documento: politicas_da_loja.pdf
Página: 3
Chunk: politicas_da_loja_p3

Score: 1.7417
Documento: politicas_da_loja.pdf
Página: 4
Chunk: politicas_da_loja_p4

Score: 1.0810
Documento: politicas_da_loja.pdf
Página: 6
Chunk: politicas_da_loja_p6

Score: 1.0724
Documento: politicas_da_loja.pdf
Página: 2
Chunk: politicas_da_loja_p2

Score: 1.0620
Documento: politicas_da_loja.pdf
Página: 7
Chunk: politicas_da_loja_p7
